In [0]:
main_path = ''

In [0]:
from pyspark.sql.functions import col, when, percentile, lower, count, avg, sum as sum_s, max as max_s, try_divide, coalesce
from pyspark.sql.types import IntegerType, DecimalType, FloatType
from pyspark.sql.window import Window

In [0]:
# Q1
# Implement: def add_valid_column(rides_df):
# Add a column:
# is_valid

# Rules:
# start_time < end_time  → True
# otherwise              → False

# Expected output:
# ride_id | start_time | end_time | is_valid

rides_df = spark.read.format('csv')\
                     .option('header','true')\
                     .load(main_path + '/rides.csv')

display(rides_df)

def add_valid_column(df):
    df = rides_df.withColumn('is_valid'
                                  ,when(col('end_time')>col('start_time')
                                       ,True
                                   )\
                                  .otherwise(False)     
                        )
    df = df.selectExpr('ride_id AS ride_id'
                      ,'start_time AS start_time'
                      ,'end_time AS  end_time'
                      ,'is_valid AS is_valid'
            )
    return df
display(add_valid_column(rides_df))

In [0]:
# temp_df = rides_df.withColumn('median', percentile(col('trip_cost'), 0.5).over(Window.partitionBy(col('ride_id'))))\
temp_df = rides_df.withColumn('trip_cost', col('trip_cost').cast(IntegerType()))
temp_ls = temp_df.approxQuantile('trip_cost', [0.25,0.5,0.75],0.05)

display(temp_ls)

q1 = temp_ls[0]
median = temp_ls[1]
q3 = temp_ls[2]
print(q1, median, q3)

In [0]:
# # Q2
# Implement:

# def remove_outliers(df):

# Use the trip_cost column.

# An observation is an outlier if:
# value < Q1 - 1.5 * IQR
# OR
# value > Q3 + 1.5 * IQR

display(rides_df)


def remove_outliers(df):
    df = df.withColumn('trip_cost', col('trip_cost').cast(IntegerType()) )
    temp_ls = df.approxQuantile('trip_cost', [0.25,0.5,0.75],0.05)
    display(temp_ls)
    q1 = temp_ls[0]
    median = temp_ls[1]
    q3 = temp_ls[2]
    iqr = q3 - q1
    print(q1, median, q3, iqr)

    df = rides_df.withColumn('outlier', 
                             when( (col('trip_cost') > (q3 + 1.5*iqr)) | 
                                   (col('trip_cost') < (q1 - 1.5*iqr))
                                  ,'Y'
                             )\
                            .otherwise('N')
                  
                  )
    return df

display(remove_outliers(rides_df))

In [0]:
# Q3 — Multiple Aggregations
# For each customer calculate:
# customer_id
# number_of_rides
# total_spend
# average_spend
# maximum_spend
# average_rating

# Only consider completed rides.

# Sort by:
# total_spend DESC
customers_df = spark.read.format('csv')\
                         .option('header','true')\
                         .load(main_path +'/customers.csv')
display(customers_df)

customer_ride_df = customers_df.join(rides_df
                                    ,rides_df.customer_id == customers_df.customer_id 
                                    ,'left'
                                )\
                               .drop(rides_df.customer_id)
display(customer_ride_df)

display(customer_ride_df.filter(lower(col('status'))=='completed').groupBy('customer_id').agg( count(col('ride_id')).alias('total_rides')
                                                                                              ,sum_s(col('trip_cost').cast(IntegerType())).alias('total_spend')
                                                                                              ,avg(col('trip_cost').cast(IntegerType())).alias('average_spend')
                                                                                              ,max_s(col('trip_cost').cast(IntegerType())).alias('max_spend')
                                                                                              ,avg(col('rating').cast(FloatType())).alias('average_rating')
                                                                                          )\
                                                                                         .orderBy(col('total_spend').desc())
)

In [0]:
#Q4 — Conditional Aggregation

# For every driver calculate:
# driver_id
# completed_rides
# cancelled_rides
# total_rides
# completion_rate

# Where:
# completion_rate =
# completed_rides / total_rides * 100

display(customer_ride_df)

display(customer_ride_df.groupBy('driver_id').agg(sum_s(when(lower(col('status')) =='completed',1).otherwise(0)).alias('completed_rides')
                                                 ,sum_s(when(lower(col('status')) =='cancelled',1).otherwise(0)).alias('cancelled_rides')
                                                 ,count(col('status')).alias('total_rides')
                                                 ,try_divide(col('completed_rides'), col('total_rides')).alias('completion_rate')
                                              )
)

In [0]:
import json
import sqlite3
import time
from datetime import datetime

queue = [
    json.dumps({
        "event_id": "e001",
        "event_time": "2026-09-30T19:00:01",
        "customer_id": 1,
        "ride_id": 2001,
        "amount": 350,
        "event_type": "ride_completed"
    }),

    json.dumps({
        "event_id": "e002",
        "event_time": "2026-09-30T19:00:08",
        "customer_id": 2,
        "ride_id": 2002,
        "amount": 480,
        "event_type": "ride_completed"
    }),

    json.dumps({
        "event_id": "e003",
        "event_time": "2026-09-30T19:00:15",
        "customer_id": 3,
        "ride_id": 2003,
        "amount": 275,
        "event_type": "ride_completed"
    }),

    json.dumps({
        "event_id": "e004",
        "event_time": "2026-09-30T19:00:25",
        "customer_id": 1,
        "ride_id": 2004,
        "amount": 525,
        "event_type": "ride_completed"
    }),

    json.dumps({
        "event_id": "e005",
        "event_time": "2026-09-30T19:00:35",
        "customer_id": 5,
        "ride_id": 2005,
        "amount": 810,
        "event_type": "ride_completed"
    })
]

print("Messages in queue:", len(queue))

In [0]:
db_path = main_path + "/ride_events_2.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS ride_events (
    event_id TEXT PRIMARY KEY,
    event_time TEXT,
    customer_id INTEGER,
    ride_id INTEGER,
    amount REAL,
    event_type TEXT
)
""")

conn.commit()

print("Database created")

cursor.execute("SELECT * FROM ride_events")
rows = cursor.fetchall()

print(rows)

In [0]:
wait_time = 10
timeout = 60

def ingest_data_into_sqlite():
    start_time = time.time()
    print(start_time)

    elapsed_time = time.time() - start_time
    while queue:
        if elapsed_time>=60:
            break
        try:
            message = queue.pop(0)
            # print(message)
            data = json.loads(message)
            # print(data)
            query = "INSERT INTO ride_events VALUES (?, ? , ?, ?, ?, ?)"
            cursor.execute(query, (data['event_id'], data['event_time'], data['customer_id'], data['ride_id'], data['amount'], data['event_type']))
            conn.commit()
            cursor.execute('SELECT * FROM ride_events')
            rows = cursor.fetchall()
            print(rows)
            time.sleep(wait_time)
        except sqlite3.Error as e:
            print(f"Database error: {e}")
    return 0
ingest_data_into_sqlite()

In [0]:
WAIT_TIME = 2
TIMEOUT = 10

def process_queue(queue, db_path, table_name):

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM ride_events")
    start_time = time.time()

    query = f"""
        INSERT INTO {table_name}
        (
            event_id,
            event_time,
            customer_id,
            ride_id,
            amount,
            event_type
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """
    print(start_time)
    try:

        while queue:

            # Check timeout
            elapsed_time = time.time() - start_time

            if elapsed_time >= TIMEOUT:
                print("Timeout reached.")
                break

            # -----------------------------
            # Read one message
            # -----------------------------

            message = queue.pop(0)

            print("Received:", message)

            # -----------------------------
            # Parse JSON
            # -----------------------------

            try:

                data = json.loads(message)

            except json.JSONDecodeError:

                print("Invalid JSON. Skipping.")
                continue

            # -----------------------------
            # Insert into SQLite
            # -----------------------------

            try:

                cursor.execute(
                    query,
                    (
                        data["event_id"],
                        data["event_time"],
                        data["customer_id"],
                        data["ride_id"],
                        data["amount"],
                        data["event_type"]
                    )
                )

                conn.commit()

                print(
                    f"Inserted event {data['event_id']}"
                )

            except sqlite3.Error as e:

                print(f"Database error: {e}")

            # -----------------------------
            # Wait before next read
            # -----------------------------

            if queue:

                print(
                    f"Waiting {WAIT_TIME} seconds..."
                )

                time.sleep(WAIT_TIME)

    finally:

        conn.close()

        print("Database connection closed.")
process_queue(queue, db_path, 'ride_events')

In [0]:
conn = sqlite3.connect(db_path)

result = conn.execute("""
    SELECT *
    FROM ride_events
    ORDER BY event_id
""").fetchall()

for row in result:
    print(row)

conn.close()